# Sector 2 — Stage 2: Q-CHAT-10 ASD Detection — Deployment-Realistic Model

**Motivation:**  
Stage 0 (94.56% acc) was inflated — `Who_completed_the_test` encoded population context:  
Health Care Professionals only test clinically-referred, suspected-ASD children → artificially high ASD rate in their subset.  
Stage 1 (83.3% acc) removed those columns but still trained on the mixed population.

**Stage 2 fix:** Filter to **parent/family-reported rows only** — the exact population the mobile app will serve.  
The model is then trained and evaluated on a sample that honestly represents what a parent filling the app will look like.

**Features:** A1–A10 (behavioural), Sex_M, Family_mem_with_ASD_Yes → **12 features**

**Pipeline:**
1. Data Preparation — population analysis, filter to Family Member rows, drop Who cols
2. Train 4 Classifiers with 5-Fold Stratified CV
3. Best Model Selection
4. Calibration Check
5. Save Trained Model
6. SHAP Analysis
7. Extract and Save Probabilities
8. Stage 0 → Stage 1 → Stage 2 Comparison

## Section 1 — Data Preparation

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix
)
from sklearn.calibration import calibration_curve
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
plt.rcParams['figure.dpi'] = 100

print('Libraries loaded successfully.')

In [ ]:
DATA_PATH = r"C:\Users\Yasindu\Desktop\Stuff\Research\Datasets\PrePROCESSED DATA\pre_processed_data_combined.csv"

data = pd.read_csv(DATA_PATH)
print(f"Full dataset shape: {data.shape}")

# Who columns present in the encoded dataset
WHO_COLS = [
    'Who_completed_the_test_Health Care Professional',
    'Who_completed_the_test_Others',
    'Who_completed_the_test_School and NGO',
]

In [ ]:
# ── Population analysis BEFORE filtering ───────────────────────────────────
# In the OHE schema (drop_first=True, alphabetical order):
#   All three Who cols = 0  →  'Family Member' (parents / relatives)  ← REFERENCE
#   Who_HCP = 1             →  'Health Care Professional'
#   Who_Others = 1          →  'Others' (self, friends, etc.)
#   Who_School = 1          →  'School and NGO'

TARGET = 'ASD_traits_Yes'

bool_cols = data.select_dtypes(include='bool').columns.tolist()
data[bool_cols] = data[bool_cols].astype(int)

def who_label(row):
    if row['Who_completed_the_test_Health Care Professional'] == 1:
        return 'Health Care Professional'
    elif row['Who_completed_the_test_Others'] == 1:
        return 'Others (self/friends)'
    elif row['Who_completed_the_test_School and NGO'] == 1:
        return 'School and NGO'
    else:
        return 'Family Member (parent/relative)'

data['_who'] = data.apply(who_label, axis=1)

pop_summary = (
    data.groupby('_who')[TARGET]
    .agg(['count', 'mean'])
    .rename(columns={'count': 'n_rows', 'mean': 'ASD_rate'})
    .sort_values('n_rows', ascending=False)
)
pop_summary['ASD_rate_%'] = (pop_summary['ASD_rate'] * 100).round(1)

print('=== ASD Rate by Who Completed the Test ===')
print(pop_summary.to_string())
print(f"\nOverall ASD rate: {data[TARGET].mean()*100:.1f}%")
print("\n→ Health Care Professionals screen clinically-referred children")
print("  → inflated ASD rate in their rows (selection bias, NOT behavioural signal)")
print("  → Family Member rows represent the app's actual user population")

In [ ]:
os.makedirs('plots', exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left — row counts by who
pop_summary['n_rows'].plot(
    kind='barh', ax=axes[0], color='steelblue', alpha=0.85
)
axes[0].set_xlabel('Number of rows', fontsize=11)
axes[0].set_title('Dataset Size by Respondent Type', fontsize=12)
axes[0].grid(axis='x', alpha=0.3)
for i, v in enumerate(pop_summary['n_rows']):
    axes[0].text(v + 20, i, str(v), va='center', fontsize=10)

# Right — ASD rate by who
colors = ['tomato' if r > 0.5 else 'steelblue'
          for r in pop_summary['ASD_rate']]
pop_summary['ASD_rate_%'].plot(
    kind='barh', ax=axes[1], color=colors, alpha=0.85
)
axes[1].axvline(data[TARGET].mean()*100, color='black',
                linestyle='--', linewidth=1.2, label='Overall mean')
axes[1].set_xlabel('ASD rate (%)', fontsize=11)
axes[1].set_title('ASD Rate by Respondent Type\n(red = above 50%)', fontsize=12)
axes[1].legend(fontsize=9)
axes[1].grid(axis='x', alpha=0.3)
for i, v in enumerate(pop_summary['ASD_rate_%']):
    axes[1].text(v + 0.5, i, f'{v}%', va='center', fontsize=10)

plt.suptitle('Population Composition — Why Stage 0 Accuracy Was Inflated',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('plots/population_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/population_analysis.png')

In [ ]:
# ── Filter to Family Member rows only ──────────────────────────────────────
# These are the rows where all Who cols = 0 (reference category = Family Member)
family_mask = (data[WHO_COLS] == 0).all(axis=1)
data_parent = data[family_mask].copy()
data_parent = data_parent.drop(columns=WHO_COLS + ['_who'])

# Remove _who from the full data too (cleanup)
data = data.drop(columns=['_who'])

print(f"Rows kept (Family Member): {len(data_parent):,}  /  {len(data):,} total  "
      f"({len(data_parent)/len(data)*100:.1f}%)")
print(f"\nClass distribution in parent-only subset:")
vc = data_parent[TARGET].value_counts()
print(f"  Non-ASD (0): {vc.get(0, 0):,}  ({vc.get(0, 0)/len(data_parent)*100:.1f}%)")
print(f"  ASD     (1): {vc.get(1, 0):,}  ({vc.get(1, 0)/len(data_parent)*100:.1f}%)")
print(f"\nRemaining features ({len(data_parent.columns)-1}):")
for col in data_parent.columns:
    if col != TARGET:
        print(f"  {col}")

In [ ]:
X = data_parent.drop(columns=[TARGET])
y = data_parent[TARGET]

print(f"Feature matrix X : {X.shape}")
print(f"Target vector  y : {y.shape}")

In [ ]:
# Stratified 80/20 split — same seed as Stage 0 / Stage 1 for consistency
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("=" * 50)
print("Train / Test Split (stratified, random_state=42)")
print("=" * 50)
print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")

vc_tr = y_train.value_counts()
print(f"\nClass balance — TRAIN:")
print(f"  Non-ASD (0): {vc_tr.get(0,0):,}  ({vc_tr.get(0,0)/len(y_train)*100:.1f}%)")
print(f"  ASD     (1): {vc_tr.get(1,0):,}  ({vc_tr.get(1,0)/len(y_train)*100:.1f}%)")

vc_te = y_test.value_counts()
print(f"\nClass balance — TEST:")
print(f"  Non-ASD (0): {vc_te.get(0,0):,}  ({vc_te.get(0,0)/len(y_test)*100:.1f}%)")
print(f"  ASD     (1): {vc_te.get(1,0):,}  ({vc_te.get(1,0)/len(y_test)*100:.1f}%)")

## Section 2 — Train 4 Classifiers with 5-Fold Stratified CV

In [ ]:
os.makedirs('results', exist_ok=True)

models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'XGBoost':             XGBClassifier(n_estimators=100, eval_metric='logloss',
                                         random_state=42, verbosity=0),
    'SVM':                 SVC(kernel='linear', probability=True, random_state=42),
}

skf     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

print('Models defined. StratifiedKFold: n_splits=5, shuffle=True, random_state=42')

In [ ]:
cv_records = {}

for name, model in models.items():
    print(f'Training {name} (5-fold CV)...')
    scores = cross_validate(
        model, X_train, y_train,
        cv=skf, scoring=scoring, return_train_score=False, n_jobs=-1
    )
    cv_records[name] = {
        'Accuracy Mean':   round(scores['test_accuracy'].mean(),  4),
        'Accuracy Std':    round(scores['test_accuracy'].std(),   4),
        'Precision Mean':  round(scores['test_precision'].mean(), 4),
        'Precision Std':   round(scores['test_precision'].std(),  4),
        'Recall Mean':     round(scores['test_recall'].mean(),    4),
        'Recall Std':      round(scores['test_recall'].std(),     4),
        'F1 Mean':         round(scores['test_f1'].mean(),        4),
        'F1 Std':          round(scores['test_f1'].std(),         4),
        'ROC-AUC Mean':    round(scores['test_roc_auc'].mean(),   4),
        'ROC-AUC Std':     round(scores['test_roc_auc'].std(),    4),
    }
    print(f'  Accuracy : {scores["test_accuracy"].mean():.4f} ± {scores["test_accuracy"].std():.4f}')
    print(f'  ROC-AUC  : {scores["test_roc_auc"].mean():.4f} ± {scores["test_roc_auc"].std():.4f}')

print('\nAll CV training complete.')

In [ ]:
cv_df = pd.DataFrame(cv_records).T.sort_values('ROC-AUC Mean', ascending=False)

print('=== 5-Fold CV Results (sorted by ROC-AUC) ===')
display(cv_df)

cv_df.to_csv('results/cv_results_qchat.csv')
print('Saved: results/cv_results_qchat.csv')

## Section 3 — Test Set Evaluation

In [ ]:
test_records  = {}
trained_models = {}

for name, model in models.items():
    print(f'Fitting {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model

    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    test_records[name] = {
        'Accuracy':  round(accuracy_score(y_test, y_pred),  4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall':    round(recall_score(y_test, y_pred),    4),
        'F1':        round(f1_score(y_test, y_pred),        4),
        'ROC-AUC':   round(roc_auc_score(y_test, y_prob),   4),
    }

test_df = pd.DataFrame(test_records).T.sort_values('ROC-AUC', ascending=False)

print('\n=== Test Set Results (sorted by ROC-AUC) ===')
display(test_df)

test_df.to_csv('results/test_results_qchat.csv')
print('Saved: results/test_results_qchat.csv')

## Section 4 — Best Model Selection

In [ ]:
best_model_name = test_df.index[0]
best_model      = trained_models[best_model_name]
best_metrics    = test_df.loc[best_model_name]

print('=' * 50)
print(f'Best Model : {best_model_name}')
print('=' * 50)
print(f'  Accuracy  : {best_metrics["Accuracy"]:.4f}')
print(f'  Precision : {best_metrics["Precision"]:.4f}')
print(f'  Recall    : {best_metrics["Recall"]:.4f}')
print(f'  F1 Score  : {best_metrics["F1"]:.4f}')
print(f'  ROC-AUC   : {best_metrics["ROC-AUC"]:.4f}')

In [ ]:
y_pred_best = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred_best)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=['Non-ASD (0)', 'ASD (1)'],
    yticklabels=['Non-ASD (0)', 'ASD (1)'],
    annot_kws={'size': 14}, ax=ax
)
ax.set_title(
    f'Confusion Matrix — {best_model_name}\n'
    f'(Parent/Family-reported rows only, test set n={len(y_test)})',
    fontsize=13, pad=10
)
ax.set_xlabel('Predicted Label', fontsize=12)
ax.set_ylabel('True Label', fontsize=12)
plt.tight_layout()
plt.savefig('plots/confusion_matrix_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/confusion_matrix_qchat.png')

## Section 5 — Calibration Check (XGBoost)

In [ ]:
xgb_model  = trained_models['XGBoost']
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]
y_test_arr = y_test.values

fraction_of_positives, mean_predicted_value = calibration_curve(
    y_test_arr, y_prob_xgb, n_bins=10
)

brier_score = float(np.mean((y_prob_xgb - y_test_arr) ** 2))

def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0.0, 1.0, n_bins + 1)
    ece  = 0.0
    n    = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        ece += mask.sum() * abs(float(y_true[mask].mean()) - float(y_prob[mask].mean()))
    return ece / n

ece = compute_ece(y_test_arr, y_prob_xgb)

print('=== XGBoost Calibration Metrics (test set, parent rows) ===')
print(f'  Brier Score : {brier_score:.4f}  (perfect = 0.0)')
print(f'  ECE         : {ece:.4f}  (perfect calibration = 0.0)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].plot([0, 1], [0, 1], 'k--', linewidth=1.2, label='Perfect calibration')
axes[0].plot(
    mean_predicted_value, fraction_of_positives,
    's-', color='steelblue', linewidth=1.8, markersize=6,
    label=f'XGBoost  (Brier={brier_score:.3f}, ECE={ece:.3f})'
)
axes[0].set_xlim(0, 1); axes[0].set_ylim(0, 1)
axes[0].set_xlabel('Mean Predicted Probability', fontsize=12)
axes[0].set_ylabel('Fraction of Positives (True ASD rate)', fontsize=12)
axes[0].set_title('Reliability Diagram', fontsize=13)
axes[0].legend(fontsize=10); axes[0].grid(alpha=0.3)

axes[1].hist(y_prob_xgb[y_test_arr == 0], bins=30, alpha=0.6,
             color='steelblue', label='Non-ASD (true=0)', density=True)
axes[1].hist(y_prob_xgb[y_test_arr == 1], bins=30, alpha=0.6,
             color='tomato',    label='ASD (true=1)',     density=True)
axes[1].axvline(0.5, color='black', linestyle='--', linewidth=1.4,
                label='Threshold = 0.5')
axes[1].set_xlabel('P(ASD)', fontsize=12)
axes[1].set_ylabel('Density', fontsize=12)
axes[1].set_title('Predicted Probability Distribution by True Class', fontsize=13)
axes[1].legend(fontsize=10); axes[1].grid(alpha=0.3)

plt.suptitle(
    f'XGBoost Calibration — Parent/Family rows   |   Brier={brier_score:.4f}   ECE={ece:.4f}',
    fontsize=13, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('plots/calibration_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/calibration_qchat.png')

## Section 6 — Save Trained Model

In [ ]:
os.makedirs('models', exist_ok=True)

print('Training final XGBoost on full parent/family subset...')
xgb_final = XGBClassifier(
    n_estimators=100, eval_metric='logloss',
    random_state=42, verbosity=0
)
xgb_final.fit(X, y)
print('Training complete.')

feature_columns = X.columns.tolist()

joblib.dump(xgb_final,       'models/xgboost_qchat_stage2.pkl')
joblib.dump(feature_columns, 'models/qchat_feature_columns.pkl')

model_size = os.path.getsize('models/xgboost_qchat_stage2.pkl')
print(f'\n  models/xgboost_qchat_stage2.pkl  ({model_size / 1024:.1f} KB)')
print(f'\nFeature columns ({len(feature_columns)}):')
for col in feature_columns:
    print(f'  {col}')

## Section 7 — SHAP Analysis

In [ ]:
print('Computing SHAP values (shap.Explainer — XGBoost 2.x compatible)...')

explainer   = shap.Explainer(xgb_final, X)
shap_values = explainer(X)

print(f'SHAP values shape: {shap_values.values.shape}')

mean_shap = pd.Series(
    np.abs(shap_values.values).mean(axis=0),
    index=feature_columns
).sort_values(ascending=False)

print('\n=== Feature Ranking by Mean |SHAP| (parent-only model) ===')
for rank, (feat, val) in enumerate(mean_shap.items(), start=1):
    print(f'  {rank:2d}. {feat:<40s} {val:.4f}')

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(
    shap_values.values, X,
    feature_names=feature_columns,
    show=False, plot_size=None
)
plt.title(
    'SHAP Feature Importance — Beeswarm\n'
    '(XGBoost Stage 2, parent/family rows, 12 features)',
    fontsize=13, pad=10
)
plt.tight_layout()
plt.savefig('plots/shap_summary_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/shap_summary_qchat.png')

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(
    shap_values.values, X,
    feature_names=feature_columns,
    plot_type='bar',
    show=False, plot_size=None
)
plt.title(
    'SHAP Mean Absolute Feature Importance\n'
    '(XGBoost Stage 2, parent/family rows, 12 features)',
    fontsize=13, pad=10
)
plt.tight_layout()
plt.savefig('plots/shap_bar_qchat.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/shap_bar_qchat.png')

### DSM-5 Mapping of Q-CHAT-10 Items

| Q-CHAT Item | DSM-5 Criterion | Behaviour Assessed |
|---|---|---|
| **A1** | A1 — Social-emotional reciprocity | Response to own name |
| **A2** | A1 — Social-emotional reciprocity | Eye contact quality |
| **A3** | A2 — Nonverbal communication | Protodeclarative pointing |
| **A4** | A2 — Nonverbal communication | Pointing to show shared interest |
| **A5** | A3 — Peer relationships | Pretend / imaginative play |
| **A6** | A2 — Nonverbal communication | Joint attention / gaze following |
| **A7** | A1 — Social-emotional reciprocity | Empathy / comfort response |
| **A8** | B — Restricted / repetitive behaviours | Typicality of first words |
| **A9** | A2 — Nonverbal communication | Use of simple gestures |
| **A10** | B — Restricted / repetitive behaviours | Visual fixation / staring |

## Section 8 — Extract and Save Probabilities

In [ ]:
os.makedirs('data', exist_ok=True)

probs = xgb_final.predict_proba(X)[:, 1]
preds = (probs >= 0.5).astype(int)

prob_df = pd.DataFrame({
    'sample_index':    np.arange(len(X)),
    'P_ASD':           probs,
    'predicted_label': preds,
    'true_label':      y.values,
})

prob_df.to_csv('data/qchat_probabilities_stage2.csv', index=False)
print(f'Saved: data/qchat_probabilities_stage2.csv  ({len(prob_df)} rows)')

display(prob_df.head(10))

print('\n--- Mean P(ASD) by true class ---')
print(
    prob_df.groupby('true_label')['P_ASD']
    .mean()
    .rename({0: 'Non-ASD (true=0)', 1: 'ASD (true=1)'})
    .to_string()
)

## Section 9 — Stage 0 → Stage 1 → Stage 2 Comparison

| Stage | Data | Features | What it shows |
|---|---|---|---|
| **Stage 0** | All 7530 rows | 15 (incl. Who cols) | Upper bound — inflated by HCP selection bias |
| **Stage 1** | All 7530 rows | 12 (behavioural + demographics) | Ablation — removes Who bias from features but not data |
| **Stage 2** | Parent rows only | 12 (behavioural + demographics) | **Deployment-realistic** — trained on actual app population |

In [ ]:
# Stage 0 and Stage 1 — hardcoded from recorded notebook outputs
stage0 = {'Accuracy': 0.9456, 'Precision': 0.9168, 'Recall': 0.9551,
           'F1': 0.9355, 'ROC-AUC': 0.9806, 'Brier': 0.0448, 'ECE': 0.0244,
           'CV Acc': 0.9334, 'CV AUC': 0.9773, 'Train rows': 6024, 'Test rows': 1506}

stage1 = {'Accuracy': 0.8327, 'Precision': 0.7885, 'Recall': 0.8138,
           'F1': 0.8009, 'ROC-AUC': 0.9151, 'Brier': 0.1172, 'ECE': 0.0527,
           'CV Acc': 0.8459, 'CV AUC': 0.9126, 'Train rows': 6024, 'Test rows': 1506}

# Stage 2 — live from this run
xgb_s2 = test_df.loc['XGBoost']
stage2 = {'Accuracy': xgb_s2['Accuracy'], 'Precision': xgb_s2['Precision'],
           'Recall': xgb_s2['Recall'], 'F1': xgb_s2['F1'], 'ROC-AUC': xgb_s2['ROC-AUC'],
           'Brier': round(brier_score, 4), 'ECE': round(ece, 4),
           'CV Acc': cv_df.loc['XGBoost', 'Accuracy Mean'],
           'CV AUC': cv_df.loc['XGBoost', 'ROC-AUC Mean'],
           'Train rows': len(X_train), 'Test rows': len(X_test)}

comp_df = pd.DataFrame({
    'Stage 0 (15 feat, all rows)':          stage0,
    'Stage 1 (12 feat, all rows)':          stage1,
    'Stage 2 (12 feat, parent rows only)':  stage2,
})

print('=== XGBoost: Three-Stage Progression ===')
display(comp_df.round(4))

comp_df.to_csv('results/stage_comparison.csv')
print('Saved: results/stage_comparison.csv')

In [ ]:
metrics = ['Accuracy', 'Precision', 'Recall', 'F1', 'ROC-AUC']
s0_vals = [stage0[m] for m in metrics]
s1_vals = [stage1[m] for m in metrics]
s2_vals = [stage2[m] for m in metrics]

x     = np.arange(len(metrics))
w     = 0.25
fig, ax = plt.subplots(figsize=(13, 6))

b0 = ax.bar(x - w,   s0_vals, w, label='Stage 0 — 15 feat, all rows\n(HCP selection bias)', color='#4C72B0', alpha=0.85)
b1 = ax.bar(x,       s1_vals, w, label='Stage 1 — 12 feat, all rows\n(bias removed from features)', color='#DD8452', alpha=0.85)
b2 = ax.bar(x + w,   s2_vals, w, label='Stage 2 — 12 feat, parent rows\n(deployment-realistic)', color='#55A868', alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(metrics, fontsize=12)
ax.set_ylim(0.75, 1.03)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(
    'Q-CHAT-10 XGBoost: Stage 0 → Stage 1 → Stage 2 Progression\n'
    'Stage 2 = deployment-realistic (parent/family reported rows only)',
    fontsize=13
)
ax.legend(fontsize=9, loc='lower right')
ax.grid(axis='y', alpha=0.3)

for bar_group in [b0, b1, b2]:
    for bar in bar_group:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.002,
                f'{h:.3f}', ha='center', va='bottom', fontsize=7.5)

plt.tight_layout()
plt.savefig('plots/stage_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: plots/stage_comparison.png')

## Final Summary

### Research Narrative

| Stage | ASD rate in data | Accuracy | ROC-AUC | Interpretation |
|---|---|---|---|---|
| Stage 0 | 41.4% (mixed populations) | ~94.6% | ~98.1% | Inflated — HCP-referred subset has near-certain ASD label |
| Stage 1 | 41.4% (mixed populations) | ~83.3% | ~91.5% | Feature ablation — Who bias removed but data still mixed |
| **Stage 2** | **parent-only ASD rate** | **TBD** | **TBD** | **Honest deployment estimate** |

### Saved Outputs

| File | Description |
|---|---|
| `plots/population_analysis.png` | ASD rate and row counts by respondent type |
| `results/cv_results_qchat.csv` | 5-fold CV for all 4 classifiers |
| `results/test_results_qchat.csv` | Test set metrics for all 4 classifiers |
| `results/stage_comparison.csv` | Stage 0 / Stage 1 / Stage 2 delta table |
| `plots/confusion_matrix_qchat.png` | Confusion matrix (best model, parent rows) |
| `plots/calibration_qchat.png` | Reliability diagram + probability distribution |
| `plots/shap_summary_qchat.png` | SHAP beeswarm (12 features, parent rows) |
| `plots/shap_bar_qchat.png` | SHAP mean absolute importance |
| `plots/stage_comparison.png` | Three-stage bar chart |
| `models/xgboost_qchat_stage2.pkl` | Production model — parent rows, 12 features |
| `models/qchat_feature_columns.pkl` | 12-feature column list for inference |
| `data/qchat_probabilities_stage2.csv` | P(ASD) for all parent-subset rows |